In [1]:
import pandas as pd
import numpy as np
import re
import string

**Data-Pre-processing**




In [2]:
df = pd.read_csv("processed_data.csv", engine="python", on_bad_lines="skip")


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1573 entries, 0 to 1572
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   label       1573 non-null   int64 
 1   subject     1571 non-null   object
 2   email_to    1565 non-null   object
 3   email_from  1573 non-null   object
 4   message     1511 non-null   object
dtypes: int64(1), object(4)
memory usage: 61.6+ KB


In [4]:
df.head()

,label,subject,email_to,email_from,message
0,1,"Generic Cialis, branded quality@",the00@speedy.uwaterloo.ca,"""Tomas Jacobs"" <RickyAmes@aol.com>",Content-Type: text/html;\nContent-Transfer-Enc...
1,0,Typo in /debian/README,debian-mirrors@lists.debian.org,Yan Morin <yan.morin@savoirfairelinux.com>,"Hi, i've just updated from the gulus and I che..."
2,1,authentic viagra,<the00@plg.uwaterloo.ca>,"""Sheila Crenshaw"" <7stocknews@tractionmarketin...","Content-Type: text/plain;\n\tcharset=""iso-8859..."
3,1,Nice talking with ya,opt4@speedy.uwaterloo.ca,"""Stormy Dempsey"" <vqucsmdfgvsg@ruraltek.com>","Hey Billy, \n\nit was really fun going out the..."
4,1,or trembling; stomach cramps; trouble in sleep...,ktwarwic@speedy.uwaterloo.ca,"""Christi T. Jernigan"" <dcube@totalink.net>",Content-Type: multipart/alternative;\n ...


In [5]:
# Total count of null values in the whole DataFrame
df.isnull().sum().sum()


np.int64(72)

In [6]:
df.isnull().sum()[df.isnull().sum() > 0] #checking for the specific column


,0
subject,2
email_to,8
message,62


In [7]:
#filling null values
df["subject"] = df["subject"].fillna("")
df["message"] = df["message"].fillna("")


In [8]:
#concatenating the column  with a space in between.
df["text"] = df["subject"].astype(str) + " " + df["message"].astype(str)


In [9]:
#A new column clean_text is created, containing the processed text
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation, numbers, etc.
    words = text.split()
    words = [lemmatizer.lemmatize(stemmer.stem(w)) for w in words if w not in stop_words]
    return " ".join(words)

df["clean_text"] = df["text"].apply(clean_text)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [10]:
#verifying
print(df[["text", "clean_text"]].head(10))

                                                text  \
0  Generic Cialis, branded quality@  Content-Type...   
1  Typo in /debian/README Hi, i've just updated f...   
2  authentic viagra Content-Type: text/plain;\n\t...   
3  Nice talking with ya Hey Billy, \n\nit was rea...   
4  or trembling; stomach cramps; trouble in sleep...   
5  Which is duty Content-Type: multipart/alternat...   
6  For Theorize Content-Type: text/plain;\n\tchar...   
7  Theorize get inside for local esc0rts who do i...   
8  Losing Weight Quickly Content-Type: text/plain...   
9  [R] Confidence-Intervals.... help... Hi...\n\n...   

                                          clean_text  
0  gener ciali brand qualiti contenttyp texthtml ...  
1  typo debianreadm hi ive updat gulu check mirro...  
2  authent viagra contenttyp textplain charsetiso...  
3  nice talk ya hey billi realli fun go night tal...  
4  trembl stomach cramp troubl sleep weak loo con...  
5  duti contenttyp multipartaltern boundarynextpa... 

In [11]:
# Remove leading/trailing spaces
df["label"] = df["label"].astype(str).str.strip()

# Keep only rows where label is 0 or 1
df = df[df["label"].isin(["0", "1"])]

# Convert to int
df["label"] = df["label"].astype(int)


In [12]:
print(df["label"].unique())
print(df.info())


[1 0]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1573 entries, 0 to 1572
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   label       1573 non-null   int64 
 1   subject     1573 non-null   object
 2   email_to    1565 non-null   object
 3   email_from  1573 non-null   object
 4   message     1573 non-null   object
 5   text        1573 non-null   object
 6   clean_text  1573 non-null   object
dtypes: int64(1), object(6)
memory usage: 86.2+ KB
None


In [13]:
#checking balanced or imbalanced
print(df["label"].value_counts())


label
1    1207
0     366
Name: count, dtype: int64


In [14]:
#Convert text into numerical features (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000)  # limit to 5000 features
X = vectorizer.fit_transform(df["clean_text"])

# Target column
y = df["label"]


In [15]:
print("Shape of TF-IDF:", X.shape)
print("Shape of labels:", y.shape)

Shape of TF-IDF: (1573, 5000)
Shape of labels: (1573,)


**Model Implementation**

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)


**Checking for imbalanced data**

In [17]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Define models
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

# Train & evaluate each model
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

    print(f"\n{name} Results:")
    print(classification_report(y_test, y_pred))

# Compare all models
import pandas as pd
results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df)



Naive Bayes Results:
              precision    recall  f1-score   support

           0       0.92      0.98      0.95       110
           1       0.99      0.98      0.98       362

    accuracy                           0.98       472
   macro avg       0.96      0.98      0.97       472
weighted avg       0.98      0.98      0.98       472


Logistic Regression Results:
              precision    recall  f1-score   support

           0       1.00      0.91      0.95       110
           1       0.97      1.00      0.99       362

    accuracy                           0.98       472
   macro avg       0.99      0.95      0.97       472
weighted avg       0.98      0.98      0.98       472


SVM Results:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99       110
           1       0.99      1.00      1.00       362

    accuracy                           0.99       472
   macro avg       1.00      0.99      0.99       472
weighted


**checking with balanced data**

In [18]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import pandas as pd

# Apply SMOTE on training data
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Define models
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

# Train & evaluate each model on balanced data
for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

    print(f"\n{name} Results:")
    print(classification_report(y_test, y_pred))

# Compare all models
results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df)



Naive Bayes Results:
              precision    recall  f1-score   support

           0       0.77      1.00      0.87       110
           1       1.00      0.91      0.95       362

    accuracy                           0.93       472
   macro avg       0.88      0.95      0.91       472
weighted avg       0.95      0.93      0.93       472


Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.98      0.97      0.98       110
           1       0.99      0.99      0.99       362

    accuracy                           0.99       472
   macro avg       0.99      0.98      0.99       472
weighted avg       0.99      0.99      0.99       472


SVM Results:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99       110
           1       0.99      1.00      1.00       362

    accuracy                           0.99       472
   macro avg       1.00      0.99      0.99       472
weighted

In [19]:
import joblib

# Save best model (let’s say SVM here)
joblib.dump(models["Logistic Regression"], "spam_model.pkl")

# Save vectorizer
joblib.dump(vectorizer, "vectorizer.pkl")


['vectorizer.pkl']

In [20]:
from google.colab import files
files.download("spam_model.pkl")
files.download("vectorizer.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
import gradio as gr
import joblib

# Load model and vectorizer
model = joblib.load("spam_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

def predict_spam(text):
    X = vectorizer.transform([text])
    pred = model.predict(X)[0]
    return "🚨 SPAM" if pred == 1 else "✅ Not Spam"

iface = gr.Interface(fn=predict_spam, inputs="text", outputs="text", title="Spam Email Detector")
iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0d4e66dd487d4daf71.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [22]:
!pip install PyPDF2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.9 MB/s eta 0:00:00


In [23]:
import gradio as gr
import joblib
import PyPDF2

# Load trained model + vectorizer
model = joblib.load("spam_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

# Function to extract text & predict
def predict_spam_from_pdf(pdf_path):
    try:
        # Read PDF
        reader = PyPDF2.PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + " "

        # If PDF is empty
        if not text.strip():
            return "❌ Could not extract text from PDF"

        # Transform and predict
        X = vectorizer.transform([text])
        pred = model.predict(X)[0]
        return "🚨 SPAM" if pred == 1 else "✅ Not Spam"

    except Exception as e:
        return f"⚠️ Error: {str(e)}"

# Gradio interface for PDF upload
iface = gr.Interface(
    fn=predict_spam_from_pdf,
    inputs=gr.File(type="filepath", label="Upload Email PDF"),
    outputs="text",
    title="📧 Spam Email Detector (PDF Upload)"
)

iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://08711e456087ad3fce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
